In [ ]:
%load_ext autoreload
%autoreload  2

from pathlib import Path
import pandas as pd
from olist_analysis import cleaning, load, metrics, checks
from olist_analysis.config import ReportConfig

cfg = ReportConfig

#ทำให้เห็น display แบบไม่โดนย่อเป็น ...
pd.set_option("display.max_colwidth", None)

# EDA  ข้อมูลรัฐทีส่งช้าและอัตราการได้ 1 ดาว

**คำถาม:** กลุ่มที่ส่งช้า P90 มีโอกาสได้ 1 ดาวกี่% เทียบกับออเดอร์อื่น และ รัฐไหนที่มีสัดส่วนการส่งช้ามากที่สุด 

**ใคร/นำไปตัดสินใจอะไร:** ข้อมูฃนี้เพื่อให้ ฝ่ายโลจิสติกส์ นำไปตัดสินใจ และ ปรับปรุงการส่งของแต่ละรัฐ

**ขอบเขต:** ข้อมูลที่ อยู่ในช่วง 2017-01-01 ไปจนถึง 2018-09-01 ที่ส่งถึงและมีการีวิวให้ดาว (โดยจะเป็นข้อมูล 1 = 1 ออเดอร์) และนับเฉพาะรัฐที่มีข้อมูล 30 แถวขึ้นไป

In [ ]:
order = load.read_table("orders", cfg.raw_dir)
order_review = load.read_table("order_reviews", cfg.raw_dir)
customer = load.read_table("customer", cfg.raw_dir)

In [ ]:
# ดูรูบร่างของข้อมูล

print(order.shape)
print(order_review.shape)
print(customer.shape)

# คอลัมไหน เป็น Unique
print((order.nunique()==len(order)))
print((order_review.nunique()==len(order_review)))
print((customer.nunique()==len(customer)))

In [ ]:
# เช็คว่า gain ของ order รีวิวต้องใช้ 2 columns ได้แก่ review order_id ประกอบกันจริงไหม

display(order_review.duplicated(["review_id", "order_id"]).sum())
display(order_review.duplicated("order_id").sum())

**สรุปจบ ขั้น 2:** order 1 แถว = 1 order_id(99,441) customer 1 แถว = 1 customer_id(99,441) 
· gain ของ order_review  column = review_id, order_id และมี 551 เกินจาก ออเดอร์เดิม -> ต้องเก็บรีวิวเดียว ต่อ ออเดอร์ก่อน merge 

In [ ]:
customer['customer_id'].nunique(), customer['customer_unique_id'].nunique()

order_per_person = customer['customer_unique_id'].value_counts()
(order_per_person >= 2).sum(),(order_per_person >=2).mean()

In [ ]:
order.columns

### ขั้นที่ 3 ตรวจสอบว่า ค่าที่จะนำมาใช้ครบถ้วยไหม โดยมี 
1. order_id ต้องไม่มีค่าว่าง
2. order_delivered_customer_date ต้องไม่มีค้่ null เพื่อดูว่า มีกี่ออเดอร์ที่ยังไม่ถึงมือลูกค้า
3. เวลาตอนสั่งและเวลาทถึงมือขนส่ง จะต้องไม่มากว่า เวลาที่ถึงมือลูกค้า
4. ค่าของ status ต้องมี ตามนี้ VALID_STATUS = ["created", "approved", "invoiced", "processing",
                "shipped", "delivered", "canceled", "unavailable"]
5. และคะแนนรีวิวในตราง order_reviews ต้องอยู่ในเวลา 1-5  


In [ ]:
VALID_STATUS = ["created", "approved", "invoiced", "processing",
                "shipped", "delivered", "canceled", "unavailable"]

order_report = checks.run_checks(order, [
            checks.Unique('order_id'),
            checks.NotNull('order_delivered_customer_date'),
            checks.DateOrder('order_purchase_timestamp', 'order_delivered_customer_date'),
            checks.DateOrder('order_delivered_carrier_date', 'order_delivered_customer_date'),
            checks.DateOrder("order_purchase_timestamp", "order_delivered_carrier_date"),
            checks.AllowedValues('order_status', VALID_STATUS)
])

display(order_report)

review_report = checks.run_checks(order_review, [
            checks.Unique('review_id','order_id'),
            checks.Unique('order_id'),
            checks.InRange('review_score', 1, 5)
])

display(review_report)

In [ ]:
clean = cleaning.clean_orders(order)
carrier_more_purchase = clean[clean['order_delivered_carrier_date'] < clean['order_purchase_timestamp']]
carrier_more_purchase[['order_id', 'order_purchase_timestamp', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'delivery_days']]


**สรุปขั้น3** จาก สิ่งที่คนพบจากสิ่งที่ลิสไว้ในโจทย์
- ค่าของ order_id ใน order ไม่มีค่าซ้ำอยู่ แต่ ใน order_review ถ้าใช้ oreder_id อย่างเดียวจะมีค่าซ้ำกัน อยู่ทื่ 537 ตัว 
- ออเดอร์ที่ยังส่งไม่ถึงลูกค้า มีจำนวน 2,965 ออดเดอร์ ห้ามนำไปคิดรวม
- เวลาที่ถึงมือรู้ค้าก่อนสั่งไม่มี แต่ มีออเดอร์ที่ส่งถึงลูกค้าก่อนขนส่งมี 23 ออกเดอร์
- เวลาที่ของถึงขนส่ง ทั้งที่ลูกค้ายังไม่ได้สั่งมี 166 ออเดอร์ ลอง ตรวจสอบเพิ่มเติมแล้ว พบว่า เกิดจากเวลา ที่ต่างกันหลักนาที อาจจะเกิดจากความผิดพลาดของระบบเวลาที่ต่างกัน ไม่กระทบกับระยะเวลาส่ง เก็บไว้ได้
- สถานะที่ต้องการมีครบ

### ขั้นที่ 4 ทำความสะอาดตาราง clean ให้เหลือแต่ แต่ review ล่าสุด 

In [ ]:
reviews_latest = (order_review
            .sort_values('review_answer_timestamp')
            .drop_duplicates('order_id', keep='last')

)




In [ ]:
clean.columns
order_review.columns
customer.columns

In [ ]:
merged = (clean
          .merge(reviews_latest[["order_id", "review_score"]],
                 on="order_id", how="left", validate="one_to_one")
          .merge(customer[["customer_id", "customer_unique_id", "customer_state"]],
                 on="customer_id", how="left", validate="one_to_one"))

len(merged)                          # ควรได้ 99,441 เท่ากับ orders
merged["review_score"].isna().sum() 

In [ ]:
steps = {'orders ทั้งหมด' : merged}

s = merged[merged['order_purchase_timestamp'].between(cfg.start, cfg.end, inclusive='left')]
steps['สั่งในช่วงขอบเขต'] = s

s = s[s['delivery_days'].notna()]
steps['ส่งถึงลูกค้าแล้ว'] = s

s = s[~s['is_date_error']]
steps['วันที่เป็นไปได้'] = s

s = s[s['review_score'].notna()]
steps['มีรีวิว'] = s

funnal = pd.Series({name:len(t) for name, t in steps.items()})

funnal

**ขั้น 4 สรุป:** 99,441 ออเดอร์ → ในขอบเขต 99,092 → ส่งถึงแล้ว 96,204
→ ตัดวันที่เป็นไปไม่ได้ 19 → มีรีวิว **95,542** (= n ของการวิเคราะห์)
ใช้รีวิวล่าสุดต่อออเดอร์ · merge ด้วย validate="one_to_one"

### ขั้นที่ 5 สำรวจ

In [ ]:
# 5.1 ตัวแปรเดียว
df['delivery_days'].quantile([0.25, 0.5, 0.75, 0.9])

# 5.2 สองตัวแปร
P90 = df['delivery_days'].quantile(0.9)
df['slow_10pct'] = df['delivery_days'] > P90
df.groupby('slow_10pct').agg(
        n = ('order_id', 'size'),
        one_star_rate = ('review_score', lambda x: (x==1).mean()),
)

In [ ]:
# 5.3 แยกตามรัฐ

by_state = df.groupby('customer_state').agg(
    n = ('order_id', 'size'),
    slow_rate = ('slow_10pct', 'mean'),
    median_day = ('delivery_days', 'median'),
    one_star_rate = ('review_score', lambda x: (x==1).mean()),
    late_rate = ('days_late', lambda x: (x>0).mean()),
)

by_state[by_state['n'] > cfg.min_n].sort_values('slow_rate', ascending=False).round(3).head(10)

In [ ]:
#พิสูทจ์ว่า ถึงแม้จะส่งเร็วก็ตามแต่ถ้าส่งเกินวันที่สัญญาก็มีโอกาสได้ 1ดว่ามากกว่า
fast = df[df["delivery_days"] < 15]
fast.groupby(fast["days_late"] > 0).agg(
    n=("order_id", "size"),
    one_star_rate=("review_score", lambda x: (x == 1).mean()),
)

จากข้อมูลที่เรามีโดยอ้างอิงตั้ง 2017-01-01 ถึง 2018-09-01  จากสมมุติฐานแรกของเรา ที่ การที่เรายิ่งส่งช้ามากเท่าไร เราก็มีโอกาส ได้ 1ดาว มากขึ้นตาม โดยได้แยกออเดอร์ที่มีการส่งนานที่สุด 10% มาสังเกต เทียบกับปกติ ก็พบว่า ออเดอร์ ใน top 10% มีโอกาสได้รับ 1 ดาว มากกว่าออเดอร์อื่นจริงโดยปกติอัตราได้ 1 ดาวจะอยู่ที่ประมาณ 6.7% แต่เป็นออเดอร์ใน 10% ที่ส่งนานที่สุด มีอัตราเพิ่มสูงขึ้นอย่างมีนัยยะ อยู่ที่ 37% เพิ่มมาถึง 5 เท่าจากปกติ ทำให้เรารู้ว่าการส่งช้าส่งผลกระทบจริงๆ ซึ่งฟังดูว่าเราเจอต้นต่อจริงตามสมมุติฐาน แต่ที่เรารู้คือส่งช้าทำให้เกิดรีวิว 1ดาวมากขึ้นเท่านั้น แล้วแค่ไหนถึงเรียกว่าช้าล่ะ เพื่อหาคำตอบ เราจึง ลองแยก top 10% ออกเป็นรัฐๆ โดยส่วนใหญ่ก็จะเป็นรัฐแถบทางเหนือที่ติดกับป่าอเมซอน เช่น AM, AP, RR หรือทางตะวันออกเฉียงเหนือ อย่างAL, PA, MA, CE, PB, SE ทั้งหมดล้วนเป็นรัฐที่ห่างใกล้จาดจุดที่ร้านค้าส่วนใหญ่ตั้งอยู่ทำให้ รัฐเหล่านี้มีจำนวนวันจัดส่งอยู่ประมาณ 18-25วัน และเราลองคำนวน อัตราการให้ 1ดาวจากรัฐที่ใกล้เหล่านี้ โดยรัฐที่ส่งมีอัตราส่งช้ามากที่สุดโดยมีอย่าง รัฐ AM ที่มีออเดอร์ที่ส่งช้าในtop10% ถึง 64%จากจำนวนออเดอร์ทั้งหมด และ รัฐ AP ที่มาอันดับ 2 มีออเดอร์ ส่งช้า 58% รัฐเหล่านี้ ที่ควรมีอัตรารีวิว 1 ดาวสูงกลับ มีอัตรารีวิว 1ดาวเพียง 6.9% และ 3.0% ตามลำดับ สิ่งพอนำไปเทียบกับค่าเฉลี่ยของ ออเดอร์ปกติแล้ว มีทั้งที่เท่าๆกัน ๆ ไปจนถึงน้อย กว่า อย่างอัตรา 1ดาว ของ AP นั้น ต่ำว่า เท่าตัว ซึ่งขัดกับสมมุติฐานที่เราคิดไว้ตอนแรก ทำให้เรามีสมมุติฐานใหม่ โดยที่ การส่งช้าของแต่ละคนก็มีบรรทัดฐานที่ไม่เท่ากัน ยิ่ง พื้นที่ห่างไกลบรรทัดฐาน ของว่าส่งช้าของเขากับคนที่อยู่ใกล้ก็ไม่น่าเท่ากัน สำหรับเขา 25วัน อาจจะเป็นเรื่องทั่วไป ดังนั้นเพื่อพิสูทจ์ สมมุติฐานนนี้ เราเลยนำ ค่าความคาดหวังที่เราตั้งไว้มาเป็นเกรณ์โดยแบ่งเป็น สิ่งทันตามกำหนด หรือเกินกำหนด และผลลัพธ์ที่ออกมาก็ทำให้สิ่งที่เราคิดนั้นมีน้ำหนักมาก ขึ้นโดยที่รัฐอย่าง AM AP AC ที่ส่งช้าเป้น อันดับ 1 2 3 มีอัตราส่งไม่ตามกำหนกที่ตั้งไว้เพียง 2.8% 3.0% 3.8% ตามลำดับ สิ่งสัมพันธ์กับ ค่าอัตราการให้1 ดาว แล้วถ้าสมมุติฐานนี้เป็นจริง ต่อให้ออเดอร์จะมีวันที่ส่งเร็วก็ตาม หากมีอัตราส่งไม่ตามกำหนดสูง อัตราการให้ 1ดาวก็ควรมากตามด้วย ลองดูตัวเลขชุดนี้ 

ทันวันสัญญา → 1 ดาว 6.0% (n = 69,162)

เลยวันสัญญา → 1 ดาว 24.4% (n = 434)

เห็นว่า พอส่งเลยกำหนด % ก็เพิ่มขึ้นสูงอย่างมีนัยยะ โดยเพิ่มขึ้นมาประมาณ 4 เท่าจากปกติสิ่งสอดคล้องกับสมมุติฐาน

ดังนั้น ก่อนที่เราจะมองว่าทำยังไงให้สินค้าเราส่งยังไงให้เร็วขึ้น เราลองมาจัดการเรื่องการตั้งค่า เวลากำหนดส่ง ให้มีความแม่นยำและเสถียรขึ้น 

ทั้งนี้ข้อสมมุติฐานที่รายงานเป็นเพียงการหาความเกี่ยวโยง ยังไม่ไช่สาเหตุที่ทำให้ได้ 1ดาวทั้งหมด ยังมีอีกหลายปัจจัยที่อาจส่งผลได้ และข้อมูลบางจุดยังน้อยเกินไปที่จะใช้ในการคาดการ และหาสาเหตุที่ชัด เราควรทดลองและหาข้อสรุปกันต่อไป 